In [6]:
import findspark
import random
import pyspark
findspark.init()


In [7]:
my_list = list(range(1, 11))  
print(my_list)  

squared_list = list(map(lambda x: x**2, my_list))
print(squared_list)  


[1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
[1, 4, 9, 16, 25, 36, 49, 64, 81, 100]


In [8]:

my_list_2 = [random.randint(1, 100) for i in range(20)] 
print(my_list_2)  # prints the list

divisible_by_5 = list(filter(lambda x: x % 5 == 0, my_list_2))  
print(divisible_by_5)  


[23, 43, 87, 32, 97, 93, 61, 49, 90, 97, 51, 72, 30, 91, 42, 90, 30, 48, 64, 71]
[90, 30, 90, 30]


## Part (a,b)

In [9]:
#creating the app builder for spark first
from pyspark.sql.functions import length, col, substring
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("UserComments").getOrCreate()
# load the file
df = spark.read.csv("user_comments.txt", header=False, inferSchema=True, sep=",")
df = df.withColumnRenamed("_c0", "UserName").withColumnRenamed("_c1", "Comment")

# find comments given by each user
df.groupBy("UserName").agg({"Comment": "count"}).show()


+--------+--------------+
|UserName|count(Comment)|
+--------+--------------+
|  Mary88|             2|
| JohnDoe|             2|
| JaneDoe|             2|
|   Ali45|             2|
|Aliya153|             1|
|   Sara2|             2|
+--------+--------------+



## Part (c)

In [10]:

# determine the number of long comments given by each user
df_filtered = df.filter(length(col("Comment")) > 20)
df_filtered.groupBy("UserName").agg({"Comment": "count"}).show()


+--------+--------------+
|UserName|count(Comment)|
+--------+--------------+
|  Mary88|             2|
| JohnDoe|             2|
| JaneDoe|             2|
|   Ali45|             1|
|Aliya153|             1|
|   Sara2|             2|
+--------+--------------+



## Part (d)

In [11]:

# count the number of UserNames starting with each English alphabet
df.groupBy(substring(col("UserName"), 0, 1)).agg({"UserName": "count"}).show()


+-------------------------+---------------+
|substring(UserName, 0, 1)|count(UserName)|
+-------------------------+---------------+
|                        M|              2|
|                        J|              4|
|                        A|              3|
|                        S|              2|
+-------------------------+---------------+



 ## Part (e)

In [12]:

# find the user who has given the maximum number of comments
df.groupBy("UserName").agg({"Comment": "count"}).orderBy(col("count(Comment)").desc()).first()


Row(UserName='Mary88', count(Comment)=2)

In [13]:
spark.stop()


## Question 4

In [15]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import explode, split, lower
from pyspark.sql.functions import regexp_replace, col
from pyspark.sql.types import StringType
from pyspark.sql.functions import broadcast

# Initialize a SparkSession
spark = SparkSession.builder.appName("StopWordsRemoval").getOrCreate()

# Read in the user comments file
comments = spark.read.text("user_comments.txt")

# Read in the stop words file
stop_words = spark.read.text("stop_words.txt").selectExpr("value as stopword")

# Broadcast the stop words file to all worker nodes
broadcast_stop_words = broadcast(stop_words)

# Split the comments into words, convert them to lowercase, and remove punctuation
words = comments.select(explode(split(col("value"), "\\s+")).alias("word"))
words = words.select(lower(regexp_replace(col("word"), "[^a-zA-Z0-9\\s]", "")).alias("word"))

# Remove stop words from the comments
words = words.join(broadcast_stop_words, words.word == broadcast_stop_words.stopword, "leftanti")
words = words.select("word")

# Count the occurrences of each word and sort by frequency
word_counts = words.groupBy("word").count().orderBy(col("count").desc())

# Show the top 10 most common words
word_counts.show(5)

# Print the total number of words in the comments file
total_words = words.count()
print("Total number of words: ", total_words)

# Print the number of unique words in the comments file
unique_words = word_counts.count()
print("Number of unique words: ", unique_words)


+-------+-----+
|   word|count|
+-------+-----+
|website|    8|
|   your|    7|
|     is|    4|
|janedoe|    2|
|   work|    2|
+-------+-----+
only showing top 5 rows

Total number of words:  66
Number of unique words:  42
